# Accessibilité — Abidjan (AMUGA), grille WorldPop

Équivalent de [`index_accessibility_notebook_def.ipynb`](https://github.com/antoinechevre/Accessibility_analysis/blob/main/index_accessibility_notebook_def.ipynb) du projet sœur [Accessibility_analysis](https://github.com/antoinechevre/Accessibility_analysis), mais pour un réseau hors de France (ici Abidjan / AMUGA) :

- **Population** : grille WorldPop sur le rectangle englobant les arrêts du GTFS (`construire_grille_population_gtfs`, cf. `world_pop_data.ipynb`) au lieu du carroyage 200x200 INSEE (France uniquement).
- **Réseau de routage** : extrait OSM téléchargé via Overpass sur cette même zone (`src/osm_extract.py`, générique — pas de dépendance à un découpage communal français) au lieu de l'extraction basée sur les codes commune INSEE.
- **Équipements (BPE)** : la Base Permanente des Équipements est un référentiel INSEE, propre à la France — pas d'équivalent branché pour la Côte d'Ivoire. Substitut : équipements OpenStreetMap pondérés par type (`extraire_amenities_osm.py` + référentiel `data/equipements_osm/Abidjan_amenities.xlsx`), dispatchés sur la grille en un score unique `land_use_data["equipements"]`, **sans découpage par domaine** (une seule source, contrairement aux domaines BPE A-G du notebook source). `cumulative_cutoff` (3.2.1), `cost_to_closest` (3.2.2), `gravity` (3.2.3), Enhanced 2SFCA (3.2.4) et la comparaison de seuils (3.2.5) sont toutes portées sur ce score. Restent **vides**, faute d'équivalent hors de France : tableaux de pôles d'équipements par domaine, benchmark par domaine, inégalités par décile de niveau de vie Filosofi (INSEE).
- Le reste (temps de trajet TC+marche via [r5py](https://r5py.readthedocs.io/)) est fonctionnel.

In [1]:
#réinitialise module

%load_ext autoreload
%autoreload 2

import os
import subprocess

# JAVA_HOME résolu dynamiquement (JDK 21 requis par r5py) : chemin fixe
# selon la machine (Temurin, Homebrew...), on essaie donc /usr/libexec/java_home
# d'abord et on ne retombe sur un chemin en dur qu'en dernier recours.
try:
    os.environ["JAVA_HOME"] = subprocess.check_output(
        ["/usr/libexec/java_home", "-v", "21"], text=True
    ).strip()
except subprocess.CalledProcessError:
    os.environ["JAVA_HOME"] = "/Library/Java/JavaVirtualMachines/temurin-21.jdk/Contents/Home"

# Fix : forcer l'initialisation de PROJ/GDAL avec les données de rasterio AVANT
# import r5py (même raison que index_accessibility_notebook_def.ipynb : le
# démarrage de la JVM par r5py écrase sinon PROJ_LIB avec un chemin invalide).
import rasterio

import r5py
import r5py.util.jvm
r5py.util.jvm.MAX_JVM_MEMORY = 2 * 1024**3  # 2 Go, à remonter si vous avez de la RAM libre

import shutil
import time
import datetime

import pandas as pd
import geopandas as gpd
import folium

import src.info_reseau as _info_reseau
from src.utils import (
    charger_gtfs,
    longueur_lignes,
    km_par_ligne_jour,
    dir_tree,
    preparer_gtfs_pour_r5py,
)
from src.hf_cache import envoyer_vers_hf, recuperer_depuis_hf
from src.worldpop import construire_grille_population_gtfs, carte_population_worldpop, zone_desservie_gtfs, pays_couverts_par_zone
from src.osm_extract import osm_pbf_creator_depuis_geofabrik
from src.utilitaires_matrix import calculer_ttm_par_lots, charger_ttm, cumulative_cutoff

In [2]:
#chemins fixes

BASE_DIR = os.getcwd()

GTFS_ZIP_PATH = os.path.join(BASE_DIR, "data", "GTFS_Africa", "Abidjan_AMUGA_GTFS_2025_mapping_v2.zip")  # <- à adapter

MEMORY_TTM_DIR = os.path.join(BASE_DIR, "data", "memory_ttm")
MEMORY_PBF_DIR = os.path.join(BASE_DIR, "data", "memory_pbf")
DOSSIER_CACHE_GTFS = os.path.join(BASE_DIR, "data", "worldpop")  # rasters pays WorldPop, gitignoré

output_path = os.path.join(BASE_DIR, "output")
data_path = os.path.join(BASE_DIR, "data")

FONDS_CARTE = {
    "OpenStreetMap": "OpenStreetMap",
    "CartoDB Positron": "CartoDB positron",
    "CartoDB Dark Matter": "CartoDB dark_matter",
}
FOND_CARTE = "CartoDB Positron"

print(BASE_DIR)
print(GTFS_ZIP_PATH)

/Users/antoinechevre/Documents/_0_Programme/6_GTFS_universal/gtfs_analysis_app
/Users/antoinechevre/Documents/_0_Programme/6_GTFS_universal/gtfs_analysis_app/data/GTFS_Africa/Abidjan_AMUGA_GTFS_2025_mapping_v2.zip


## Charger le GTFS

In [3]:
#charge GTFS en feed et nom du réseau

feed = charger_gtfs(GTFS_ZIP_PATH)
print(type(feed))

nb_agences = len(feed.agency)
if nb_agences > 3:
    print(f"⚠ Ce GTFS regroupe {nb_agences} agences : ce que l'app ne peut pas gérer. Charger un GTFS urbain uniquement.")

# On appelle via le module (pas le nom importé) pour ne jamais l'écraser avec
# son résultat : sinon, réexécuter cette cellule une deuxième fois lève
# "TypeError: 'str' object is not callable".
nom_reseau_str = _info_reseau.nom_reseau_str(feed)
print(nom_reseau_str)

dates_service_list, date_debut, date_fin, date_JOB = _info_reseau.dates_service(feed)
print(dates_service_list, date_debut, date_fin, date_JOB)

# Véhicules.km pour la date JOB (générique, indépendant de la BPE)
longueur_par_ligne = longueur_lignes(feed)
vkm_par_ligne_job = km_par_ligne_jour(feed, longueur_par_ligne, date_JOB)
total_vkm_job = vkm_par_ligne_job["total_km"].sum()
print(f"Véhicules.km le {date_JOB} (JOB) : {total_vkm_job:,.0f} km, sur {len(vkm_par_ligne_job)} ligne(s)")
vkm_par_ligne_job.sort_values("total_km", ascending=False)

Chargement du fichier GTFS : /Users/antoinechevre/Documents/_0_Programme/6_GTFS_universal/gtfs_analysis_app/data/GTFS_Africa/Abidjan_AMUGA_GTFS_2025_mapping_v2.zip
✓ GTFS chargé avec succès
<class 'gtfs_kit.feed.Feed'>
⚠ Ce GTFS regroupe 8 agences : ce que l'app ne peut pas gérer. Charger un GTFS urbain uniquement.
AMUGA - Gbaka 10 à 14 places - Gbaka 18 à 22 places - Gbaka 26 à 32 places - Gba…
['20250501', '20250502', '20250503', '20250504', '20250505', '20250506', '20250507', '20250508', '20250509', '20250510', '20250511', '20250512', '20250513', '20250514', '20250515', '20250516', '20250517', '20250518', '20250519', '20250520', '20250521', '20250522', '20250523', '20250524', '20250525', '20250526', '20250527', '20250528', '20250529', '20250530', '20250531', '20250601', '20250602', '20250603', '20250604', '20250605', '20250606', '20250607', '20250608', '20250609', '20250610', '20250611', '20250612', '20250613', '20250614', '20250615', '20250616', '20250617', '20250618', '20250619', 

,route_id,total_km,date
179,Gare de bassam vers gare bonoua,625.299052,20281228
633,rond point samake abobo vers Nouvelle gare de ...,533.905890,20281228
164,Gare bonoua vers village hono,515.129441,20281228
183,Gare jacqueville apache vers treichville avenu...,508.888983,20281228
363,adjamé mosquée vers gare jacqueville,493.222308,20281228
...,...,...,...
386,attecoube vers mosquée adjame,7.004424,20281228
324,abobo anokoua-pk18 abobo,5.388920,20281228
377,angre petro ivoire cocody vers terminus 81_82 ...,3.976486,20281228
344,académie yopougon vers fin goudron yopougon,3.647674,20281228


## Construction de la grille de population (WorldPop)

Même zone que `world_pop_data.ipynb` : rectangle englobant tous les arrêts du GTFS, avec une marge de sécurité `MARGE_KM`. `population_grid_agglo`/`land_use_data` (mêmes noms que le notebook source) servent de grille de référence pour toute la suite — origines/destinations du routage, "opportunité" population pour `cumulative_cutoff`.

In [4]:
MARGE_KM = 5  # marge de sécurité ajoutée sur chaque côté du rectangle englobant les arrêts, en km
ANNEE_GTFS = 2020  # dernière année disponible dans le dataset WorldPop "Unconstrained individual countries" (2000-2020)
RESOLUTION_M_GTFS = 800  # taille de carreau cible en mètres — 800 plutôt que 400 (cf. index_accessibility_notebook_africa.ipynb) : grille ~4x plus petite (aire du carreau x4), donc TTM ~16x plus légère, pour tenir en mémoire au rechargement (charger_ttm) sur une machine à RAM limitée — cf. l'OOM observé avec la grille 400m (86,9 Go de mémoire pour un kernel tué par macOS, alors que la machine n'a que 16 Go de RAM) ; estimé ~5,4 Go ici, confortable.

population_grid_agglo, lat_centre_gtfs, lon_centre_gtfs = construire_grille_population_gtfs(
    feed,
    marge_km=MARGE_KM,
    annee=ANNEE_GTFS,
    resolution_m=RESOLUTION_M_GTFS,
    dossier_cache=DOSSIER_CACHE_GTFS,
)

NOM_ZONE_GTFS = os.path.splitext(os.path.basename(GTFS_ZIP_PATH))[0]

# land_use_data : GeoDataFrame de travail pour cumulative_cutoff etc. (mêmes
# noms de colonnes que le notebook source : id, population)
land_use_data = population_grid_agglo[["id", "population"]].copy()

print(f"{len(population_grid_agglo)} carreaux, population totale : {population_grid_agglo['population'].sum():,.0f} habitants")
population_grid_agglo.head()

✓ Zone GTFS : 1116 arrêts, rectangle centré sur 5.4048, -4.0618
✓ Pays couverts par la zone GTFS : ['CIV']
✓ Raster déjà en cache : /Users/antoinechevre/Documents/_0_Programme/6_GTFS_universal/gtfs_analysis_app/data/worldpop/CIV_ppp_2020.tif
✓ CIV : 11924 carreaux dans la zone
11924 carreaux, population totale : 6,151,695 habitants


,population,geometry,id
0,21.021847,"POLYGON ((-4.76542 5.68625, -4.76542 5.67875, ...",0
1,19.696985,"POLYGON ((-4.75792 5.68625, -4.75792 5.67875, ...",1
2,21.149935,"POLYGON ((-4.75042 5.68625, -4.75042 5.67875, ...",2
3,20.996647,"POLYGON ((-4.74292 5.68625, -4.74292 5.67875, ...",3
4,24.321194,"POLYGON ((-4.73542 5.68625, -4.73542 5.67875, ...",4


## Extraction OSM pour le réseau de routage

Même rectangle (arrêts GTFS + marge) que la grille de population ci-dessus, pour que le réseau routier couvre au moins toute la zone analysée.

Extraits pays pré-construits Geofabrik (`osm_pbf_creator_depuis_geofabrik`, un fichier par pays de `pays_couverts_par_zone(zone_geom)`), découpés précisément au contour de la zone avec osmium — plutôt que l'API Overpass tuile par tuile (`osm_pbf_creator_depuis_geometrie`, cf. `src/osm_extract.py`) : cette dernière plantait systématiquement ici (`overpass-api.de` et plusieurs miroirs indépendants tous injoignables, alors que `download.geofabrik.de` répond normalement — probable filtrage réseau local ciblant spécifiquement les instances Overpass). Avantage annexe, indépendant du contexte réseau : un seul gros fichier par pays, pas de risque de géométrie tronquée aux frontières de tuile (testé sur Abidjan : `osmium check-refs` ne signale aucun nœud manquant, contre le retry systématique qu'il fallait parfois avec Overpass).

⚠ Premier appel : peut prendre plusieurs minutes (téléchargement de l'extrait pays + osmium). Extrait pays mis en cache localement (`data/osm_pbf_pays/`, réutilisable pour n'importe quelle zone du même pays) et résultat final mis en cache localement (`data/memory_pbf/`) et sur Hugging Face (`memory_pbf/agglo_osm_pbf_{nom_reseau_str}.osm.pbf`, namespacé par réseau) : les appels suivants sont immédiats.

In [5]:
zone_geom, _, _ = zone_desservie_gtfs(feed, marge_km=MARGE_KM)

os.makedirs(MEMORY_PBF_DIR, exist_ok=True)
PBF_PATH_SAVED = os.path.join(MEMORY_PBF_DIR, f"agglo_osm_pbf_{nom_reseau_str}.osm.pbf")
recuperer_depuis_hf(f"memory_pbf/agglo_osm_pbf_{nom_reseau_str}.osm.pbf", PBF_PATH_SAVED)

OSM_WORK_DIR = os.path.join(data_path, "osm_extract")
AGGLO_PBF_PATH = os.path.join(OSM_WORK_DIR, "agglo.osm.pbf")

if os.path.exists(PBF_PATH_SAVED):
    os.makedirs(OSM_WORK_DIR, exist_ok=True)
    shutil.copyfile(PBF_PATH_SAVED, AGGLO_PBF_PATH)
    print(f"extrait OSM déjà présent pour ce réseau, copié depuis {PBF_PATH_SAVED}")
else:
    codes_pays = pays_couverts_par_zone(zone_geom)
    print(f"✓ Pays couverts par la zone GTFS (Geofabrik) : {sorted(codes_pays)}")
    AGGLO_PBF_PATH = osm_pbf_creator_depuis_geofabrik(zone_geom, OSM_WORK_DIR, codes_pays)
    shutil.copyfile(AGGLO_PBF_PATH, PBF_PATH_SAVED)
    envoyer_vers_hf(PBF_PATH_SAVED, f"memory_pbf/agglo_osm_pbf_{nom_reseau_str}.osm.pbf")

print(AGGLO_PBF_PATH)

extrait OSM déjà présent pour ce réseau, copié depuis /Users/antoinechevre/Documents/_0_Programme/6_GTFS_universal/gtfs_analysis_app/data/memory_pbf/agglo_osm_pbf_AMUGA - Gbaka 10 à 14 places - Gbaka 18 à 22 places - Gbaka 26 à 32 places - Gba….osm.pbf
/Users/antoinechevre/Documents/_0_Programme/6_GTFS_universal/gtfs_analysis_app/data/osm_extract/agglo.osm.pbf


## Équipements (substitut OSM pondéré à la BPE)

Pas de Base Permanente des Équipements pour la Côte d'Ivoire : extraction de tous les `amenity=*` OpenStreetMap, pondérés par type via le référentiel défini à la main sur Abidjan (`data/equipements_osm/Abidjan_amenities.xlsx`).

Extraction locale (`extraire_amenities_depuis_pbf`, osmium `tags-filter` + `export`) à partir d'`AGGLO_PBF_PATH` — le `.osm.pbf` déjà téléchargé et découpé sur la zone pour le réseau de routage, cellule précédente — plutôt que `extraire_amenities_osm.py` (API Overpass) : même souci d'indisponibilité d'Overpass que pour le réseau de routage (cf. cellule précédente), et ça évite en plus une deuxième requête réseau sur la même zone, le pbf de routage couvrant déjà tout ce qu'il faut.

In [6]:
from src.osm_extract import extraire_amenities_depuis_pbf

# Substitut à la BPE (pas de données INSEE pour la Côte d'Ivoire) : tous les
# équipements OpenStreetMap taggés amenity=* sur AGGLO_PBF_PATH (déjà
# téléchargé/découpé sur zone_geom pour le réseau de routage, cellule
# précédente — extraction locale via osmium, pas de requête réseau ici),
# pondérés par type d'équipement — pondération définie à la main sur Abidjan
# (data/equipements_osm/Abidjan_amenities.xlsx, feuille "resume_par_type",
# colonne "Ponderation" : 0 = pas un pôle d'équipement pertinent, jusqu'à 30
# pour les équipements structurants comme hôpital/gouvernement/bâtiment
# public) et réutilisée telle quelle comme référentiel unique pour toutes
# les villes — un type d'amenity absent de ce référentiel (non rencontré sur
# Abidjan) reçoit une pondération de 0, pas d'erreur.

DOSSIER_EQUIPEMENTS_OSM = os.path.join(data_path, "equipements_osm")
PONDERATION_XLSX = os.path.join(DOSSIER_EQUIPEMENTS_OSM, "Abidjan_amenities.xlsx")  # référentiel partagé, cf. ci-dessus

os.makedirs(DOSSIER_EQUIPEMENTS_OSM, exist_ok=True)
NOM_VILLE_SIMPLE = NOM_ZONE_GTFS.split("_")[0]
CHEMIN_EQUIPEMENTS_GPKG = os.path.join(DOSSIER_EQUIPEMENTS_OSM, f"{NOM_VILLE_SIMPLE.lower()}_equipements.gpkg")

# Cache-first (comme memory_pbf/memory_ttm plus haut) : si ce réseau a déjà
# été traité (par ce run ou un run précédent, ici ou sur un autre
# déploiement), le gpkg pondéré est repris tel quel sur Hugging Face
# (equipements_osm/{ville}_equipements.gpkg) — évite de dépendre du
# référentiel PONDERATION_XLSX (fichier édité à la main, pas toujours
# présent en local) juste pour relire un résultat déjà calculé.
if recuperer_depuis_hf(f"equipements_osm/{NOM_VILLE_SIMPLE.lower()}_equipements.gpkg", CHEMIN_EQUIPEMENTS_GPKG):
    amenities_geo = gpd.read_file(CHEMIN_EQUIPEMENTS_GPKG)
    print(f"✓ Équipements repris du cache : {CHEMIN_EQUIPEMENTS_GPKG} ({len(amenities_geo)} amenity(s))")
else:
    amenities = extraire_amenities_depuis_pbf(AGGLO_PBF_PATH, OSM_WORK_DIR)

    ponderation_par_amenity = pd.read_excel(PONDERATION_XLSX, sheet_name="resume_par_type").set_index("amenity")["Ponderation"]
    amenities["ponderation"] = amenities["amenity"].map(ponderation_par_amenity).fillna(0)

    nb_hors_referentiel = amenities.loc[~amenities["amenity"].isin(ponderation_par_amenity.index), "amenity"].nunique()
    print(f"✓ {len(amenities)} amenity(s) extrait(s), {(amenities['ponderation'] > 0).sum()} avec une pondération > 0")
    if nb_hors_referentiel:
        print(f"  ({nb_hors_referentiel} type(s) d'amenity absent(s) du référentiel Abidjan, pondérés à 0 par défaut)")

    amenities_geo = gpd.GeoDataFrame(
        amenities, geometry=gpd.points_from_xy(amenities["lon"], amenities["lat"]), crs="EPSG:4326",
    )
    amenities_geo.to_file(CHEMIN_EQUIPEMENTS_GPKG, driver="GPKG")
    envoyer_vers_hf(CHEMIN_EQUIPEMENTS_GPKG, f"equipements_osm/{NOM_VILLE_SIMPLE.lower()}_equipements.gpkg")
    print(f"✓ Équipements sauvegardés : {CHEMIN_EQUIPEMENTS_GPKG}")

amenities_geo.loc[amenities_geo["ponderation"] > 0].groupby("amenity")["ponderation"].first().sort_values(ascending=False)

✓ Équipements repris du cache : /Users/antoinechevre/Documents/_0_Programme/6_GTFS_universal/gtfs_analysis_app/data/equipements_osm/abidjan_equipements.gpkg (19734 amenity(s))


amenity
public_building       30.0
government            30.0
hospital              30.0
university            20.0
townhall              20.0
courthouse            20.0
library               10.0
language_school       10.0
transportation        10.0
social_centre         10.0
research_institute    10.0
marketplace           10.0
cinema                10.0
veterinary            10.0
ferry_terminal        10.0
events_venue          10.0
conference_centre     10.0
community_centre      10.0
college               10.0
clinic                10.0
pharmacy               4.0
prep_school            2.0
doctors                2.0
dentist                2.0
school                 2.0
bank                   2.0
post_office            1.0
social_facility        1.0
Name: ponderation, dtype: float64

## Dispatch pondéré des équipements sur la grille (scoring BPE)

Jointure spatiale des équipements extraits ci-dessus dans les carreaux 800x800m de `population_grid_agglo`, pondérés par type (`land_use_data["equipements"]` = somme des pondérations, pas un simple comptage).

In [7]:
# Dispatch pondéré des équipements sur les carreaux 800x800m de
# population_grid_agglo (jointure spatiale point-dans-polygone) :
# land_use_data["equipements"] est désormais une somme pondérée par carreau
# (le "scoring BPE" de ce notebook), pas un simple comptage isopondéré.
jointure = gpd.sjoin(amenities_geo, population_grid_agglo[["id", "geometry"]], how="inner", predicate="within")
score_par_carreau = jointure.groupby("id")["ponderation"].sum().rename("equipements")

land_use_data = land_use_data.merge(score_par_carreau, on="id", how="left")
land_use_data["equipements"] = land_use_data["equipements"].fillna(0)

print(f"✓ score total : {land_use_data['equipements'].sum():,.0f} (pondéré), sur {len(land_use_data)} carreaux")
land_use_data.sort_values("equipements", ascending=False).head()

✓ score total : 22,443 (pondéré), sur 11924 carreaux


,id,population,equipements
9062,9062,3087.347656,293.0
8134,8134,5189.927734,243.0
8504,8504,3922.566650,227.0
8149,8149,1976.509521,224.0
9417,9417,14566.200195,205.0


## Construction du réseau de transport multimodal (r5py)

Équivalent de `setup_r5(data_path)` : l'objet `TransportNetwork` joue à la fois le rôle du réseau construit et du point d'entrée pour les calculs de temps de trajet (`TravelTimeMatrix`).

`allow_errors=True` : ce GTFS contient des `stop_times.txt` corrompus sur certains trips (ex. "Abobo Gare abobo- Alepe Gare Alepe_dir0_morning_peak" : `arrival_time` qui grimpe jusqu'à `146:19:41`, soit >72h — vraisemblablement un bug de l'outil qui a généré ce GTFS, ~344 lignes concernées sur 102 713). R5 rejette par défaut tout le fichier dès qu'une valeur dépasse son garde-fou de 72h (`RangeError: ... outside of acceptable range [0.0,72.0]`) ; `allow_errors=True` lui fait ignorer les entrées fautives et charger le reste du réseau normalement, plutôt que de corriger le GTFS à la main.

In [8]:
print(data_path)
dir_tree(data_path)

GTFS_PATH_R5PY = preparer_gtfs_pour_r5py(GTFS_ZIP_PATH)

try:
    transport_network = r5py.TransportNetwork(
        osm_pbf=str(AGGLO_PBF_PATH),
        gtfs=[str(GTFS_PATH_R5PY)],
        allow_errors=True,
    )
except Exception:
    # r5py met en cache dans Config().CACHE_DIR (~/.cache/r5py) le graphe déjà
    # construit ET une copie de travail de l'OSM pbf : un run précédent
    # interrompu peut y laisser un fichier à moitié écrit, qui ne se répare
    # jamais tout seul (cf. index_accessibility_notebook_def.ipynb, même
    # correctif). On vide ce cache (SAUF les .jar R5, ~65 Mo, déjà
    # téléchargés une fois pour toutes) et on relance une fois.
    from r5py.util import Config

    cache_dir = Config().CACHE_DIR
    for entree in cache_dir.iterdir():
        if entree.suffix == ".jar":
            continue
        if entree.is_dir():
            shutil.rmtree(entree, ignore_errors=True)
        else:
            entree.unlink(missing_ok=True)
    transport_network = r5py.TransportNetwork(
        osm_pbf=str(AGGLO_PBF_PATH),
        gtfs=[str(GTFS_PATH_R5PY)],
        allow_errors=True,
    )

population_grid_agglo.info()

/Users/antoinechevre/Documents/_0_Programme/6_GTFS_universal/gtfs_analysis_app/data
├── .cache
│   └── huggingface
│       ├── download
│       │   ├── GTFS
│       │   │   ├── AMUGA_GTFS_2025_mapping_v2.zip.lock
│       │   │   ├── AMUGA_GTFS_2025_mapping_v2.zip.metadata
│       │   │   ├── Amsterdam_gtfs.zip.lock
│       │   │   ├── Amsterdam_gtfs.zip.metadata
│       │   │   ├── Antwerp_gtfs.zip.lock
│       │   │   ├── Antwerp_gtfs.zip.metadata
│       │   │   ├── Barcelona_mdb-3232-202607230126.zip.lock
│       │   │   ├── Barcelona_mdb-3232-202607230126.zip.metadata
│       │   │   ├── Berlin_gtfs_merge.zip.lock
│       │   │   ├── Berlin_gtfs_merge.zip.metadata
│       │   │   ├── Bogota_gtfs.zip.lock
│       │   │   ├── Bogota_gtfs.zip.metadata
│       │   │   ├── Brussels_gtfs.zip.lock
│       │   │   ├── Brussels_gtfs.zip.metadata
│       │   │   ├── Chicago_google_transit.zip.lock
│       │   │   ├── Chicago_google_transit.zip.metadata
│       │   │   ├── Copenhagen_gtfs.zip

/Users/antoinechevre/Documents/_0_Programme/6_GTFS_universal/gtfs_analysis_app/env/lib/python3.12/site-packages/r5py/r5/transport_network.py:266: RuntimeWarning: R5 reported the following issues with GTFS file Abidjan_AMUGA_GTFS_2025_mapping_v2.zip: 
RangeError: stop_times line 23821: Number 107.0 outside of acceptable range [0.0,72.0].
- RangeError: stop_times line 53428: Number 126.0 outside of acceptable range [0.0,72.0].
- RangeError: stop_times line 34664: Number 124.0 outside of acceptable range [0.0,72.0].
- RangeError: stop_times line 34658: Number 106.0 outside of acceptable range [0.0,72.0].
- RangeError: stop_times line 53419: Number 87.0 outside of acceptable range [0.0,72.0].
- RangeError: stop_times line 23826: Number 126.0 outside of acceptable range [0.0,72.0].
- RangeError: stop_times line 34655: Number 82.0 outside of acceptable range [0.0,72.0].
- RangeError: stop_times line 23823: Number 120.0 outside of acceptable range [0.0,72.0].
- RangeError: stop_times line 346

## Matrice de temps de trajet (TTM)

Origines/destinations = centroïdes de la grille de population. Calculée par lots (`calculer_ttm_par_lots`) pour borner le pic mémoire, écrite sur disque au fur et à mesure. Mise en cache localement et sur Hugging Face (`memory_ttm/ttm_{nom_reseau_str}_{RESOLUTION_M_GTFS}m.parquet` — suffixe de résolution pour ne jamais confondre avec la grille 400m de `index_accessibility_notebook_africa.ipynb`, cf. cellule suivante), rechargée directement si calculée il y a moins de 10 jours.

In [9]:
points = population_grid_agglo[["id", "geometry"]].copy()
points["geometry"] = points.geometry.centroid

os.makedirs(MEMORY_TTM_DIR, exist_ok=True)
# Suffixe _{RESOLUTION_M_GTFS}m : la TTM dépend de la résolution de la grille
# (nombre et identité des carreaux), contrairement au pbf routier ou aux
# équipements (indépendants de la résolution, cf. cellules précédentes) — sans
# ce suffixe, ce notebook retomberait sur le même nom de fichier que
# index_accessibility_notebook_africa.ipynb (grille 400m), en local comme sur
# HF, et rechargerait silencieusement la mauvaise matrice (mauvaise grille).
TTM_PATH = os.path.join(MEMORY_TTM_DIR, f"ttm_{nom_reseau_str}_{RESOLUTION_M_GTFS}m.parquet")

recuperer_depuis_hf(f"memory_ttm/ttm_{nom_reseau_str}_{RESOLUTION_M_GTFS}m.parquet", TTM_PATH)

ttm_cache_recent = (
    os.path.exists(TTM_PATH) and (time.time() - os.path.getmtime(TTM_PATH)) < 10 * 24 * 3600
)

if ttm_cache_recent:
    ttm = charger_ttm(TTM_PATH)
    print(f"ttm rechargé depuis le cache (< 10 jours) : {TTM_PATH}")
else:
    departure_datetime = datetime.datetime.strptime(date_JOB, "%Y%m%d").replace(
        hour=14, minute=0, second=0
    )

    calculer_ttm_par_lots(
        r5py,
        transport_network,
        points,
        departure=departure_datetime,
        transport_modes=[r5py.TransportMode.WALK, r5py.TransportMode.TRANSIT],
        max_time_walking=datetime.timedelta(minutes=30),
        max_time=datetime.timedelta(minutes=120),
        ttm_path=TTM_PATH,
        on_step=print,
    )
    ttm = charger_ttm(TTM_PATH)

    envoyer_vers_hf(TTM_PATH, f"memory_ttm/ttm_{nom_reseau_str}_{RESOLUTION_M_GTFS}m.parquet")

ttm.head()

/var/folders/8x/96rx2t6n4ws9jpgs900bdwfw0000gn/T/ipykernel_85880/3225264955.py:2: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  points["geometry"] = points.geometry.centroid
/Users/antoinechevre/Documents/_0_Programme/6_GTFS_universal/gtfs_analysis_app/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[hf_cache] recuperer_depuis_hf('memory_ttm/ttm_AMUGA - Gbaka 10 à 14 places - Gbaka 18 à 22 places - Gbaka 26 à 32 places - Gba…_800m.parquet') absent de antoinechevre/accessibility-data : RemoteEntryNotFoundError('404 Client Error. (Request ID: Root=1-6a8ddf59-6e1c0217745d0003569a83df;12bec524-fb09-4a8f-80db-e601c4d3204b)\n\nEntry Not Found for url: https://huggingface.co/datasets/antoinechevre/accessibility-data/resolve/main/memory_ttm/ttm_AMUGA%20-%20Gbaka%2010%20%C3%A0%2014%20places%20-%20Gbaka%2018%20%C3%A0%2022%20places%20-%20Gbaka%2026%20%C3%A0%2032%20places%20-%20Gba%E2%80%A6_800m.parquet.')
[hf_cache] recuperer_depuis_hf('memory_ttm/ttm_AMUGA - Gbaka 10 à 14 places - Gbaka 18 à 22 places - Gbaka 26 à 32 places - Gba…_800m.parquet') absent de antoinechevre/ww_GTFS : RemoteEntryNotFoundError('404 Client Error. (Request ID: Root=1-6a8ddf59-6d8640873548f6113eff44b0;7b979394-e183-452b-b07e-343ebfaa105e)\n\nEntry Not Found for url: https://huggingface.co/datasets/antoinechevre/ww_GT

Processing Files (1 / 1): 100%|██████████|  259MB /  259MB, 21.5MB/s  
New Data Upload: 100%|██████████| 66.3MB / 66.3MB, 5.50MB/s  


,from_id,to_id,travel_time
0,0,0,0.0
1,0,1,NaN
2,0,2,NaN
3,0,3,NaN
4,0,4,NaN


## 3.2.1 Mesure des opportunités cumulées (`cumulative_cutoff`)

Seule mesure du notebook source qui ne dépend pas d'une source d'équipements : `opportunity="population"` mesure le nombre d'habitants accessibles en <= `cutoff` minutes depuis chaque carreau. 3.2.2 à 3.2.5 sont portées plus bas avec `opportunity="equipements"` (score OSM pondéré, cf. section "Dispatch pondéré des équipements sur la grille" plus haut) — sans découpage par domaine (une seule source d'équipements pour la Côte d'Ivoire, contrairement à la BPE du notebook source).

In [10]:
cum_opportunities = cumulative_cutoff(
    ttm,
    land_use_data=land_use_data,
    opportunity="population",
    travel_cost="travel_time",
    cutoff=30,
)

cum_opportunities.head()

,id,population
0,0,21
1,1,19
2,2,21
3,3,45
4,4,119


## 3.2.2 Coût de trajet minimum (`cost_to_closest`)

Contrairement au notebook source (domaine BPE `DOMAINE_CIBLE`, `land_use_data_domaine`), il n'y a ici qu'un seul score d'équipements, tous types OSM confondus (`land_use_data["equipements"]`, cellule "Dispatch pondéré..." ci-dessus) — pas de découpage par domaine pour la Côte d'Ivoire. On passe donc `land_use_data` directement à `cost_to_closest` (`opportunity="equipements"`), ce qui court-circuite la construction automatique par domaine BPE : les 4 premiers paramètres (`land_use_data_domaine`, `BPE_agglo`, `_land_use_data_global`, `DOMAINES_BPE`) ne sont alors jamais utilisés, on peut leur passer `None`.

In [11]:
from src.utilitaires_matrix import cost_to_closest

min_time_equipements = cost_to_closest(
    None, None, None, {},  # land_use_data_domaine, BPE_agglo, _land_use_data_global, DOMAINES_BPE : jamais utilisés puisque land_use_data est fourni directement ci-dessous (DOMAINES_BPE reste un dict vide, pas None : cost_to_closest y fait DOMAINES_BPE.get(...) même dans ce cas, pour le message affiché)
    ttm,
    opportunity="equipements",
    travel_cost="travel_time",
    land_use_data=land_use_data,
    n=1,  # atteint dès qu'un carreau de score pondéré >= 1 est accessible (pas de notion de "pôle" par domaine ici)
)

min_time_equipements.head()

min_time calculé pour accéder à : equipements


,id,travel_time
0,0,inf
1,1,inf
2,2,inf
3,3,inf
4,4,inf


## 3.2.3 Mesures de gravité (`gravity`)

Comme pour 3.2.2 : pas de domaine BPE, `opportunity="equipements"` directement sur `land_use_data` (score OSM pondéré, tous types confondus) — `gravity()` n'a de toute façon aucune dépendance à la BPE dans sa signature, contrairement à `cost_to_closest`.

In [12]:
from src.utilitaires_matrix import decay_exponential, gravity

negative_exp_grav = gravity(
    ttm,
    land_use_data=land_use_data,
    opportunity="equipements",
    travel_cost="travel_time",
    decay_function=decay_exponential(0.2),
)

negative_exp_grav.head()

,id,equipements
0,0,0.0
1,1,0.0
2,2,0.0
3,3,0.0
4,4,0.0


## 3.2.4 Mesures de compétition (Enhanced 2SFCA)

Pas d'implémentation fidèle du BFCA (Paez, Higgins & Vivona 2019, algorithme d'équilibrage itératif offre/demande, trop spécifique pour être reconstitué de mémoire) — Enhanced 2SFCA (Luo & Qi, 2009) à la place, comme dans le notebook source : ratio offre/demande pondéré par la décroissance, en deux étapes, sans boucle d'équilibrage. Les valeurs ne correspondent pas exactement à `{accessibility}::floating_catchment_area(method = "bfca")`.

`supply` = `equipements`, `demand` = `population`, tous deux déjà dans `land_use_data` (pas de `land_use_data_domaine` ni de merge à faire, contrairement au notebook source). `enhanced_2sfca_par_lots()` plutôt que `enhanced_2sfca()` : relit `ttm` depuis `TTM_PATH` par row group (pyarrow) plutôt que de dupliquer la matrice en mémoire via deux `.merge()` — moins critique ici sur la grille WorldPop 800m d'Abidjan (~16x plus légère que la grille 400m) qu'à l'origine sur Lyon/TCL, mais résultat identique et sans coût supplémentaire.

In [13]:
from src.utilitaires_matrix import enhanced_2sfca_par_lots

e2sfca_equipements = enhanced_2sfca_par_lots(
    TTM_PATH,
    land_use_data=land_use_data,
    opportunity="equipements",
    travel_cost="travel_time",
    demand="population",
    decay_function=decay_exponential(0.05),
    on_step=print,
)

e2sfca_equipements.head()

Enhanced 2SFCA, passe 1/2 (demande)... lot 1/143
Enhanced 2SFCA, passe 1/2 (demande)... lot 2/143
Enhanced 2SFCA, passe 1/2 (demande)... lot 3/143
Enhanced 2SFCA, passe 1/2 (demande)... lot 4/143
Enhanced 2SFCA, passe 1/2 (demande)... lot 5/143
Enhanced 2SFCA, passe 1/2 (demande)... lot 6/143
Enhanced 2SFCA, passe 1/2 (demande)... lot 7/143
Enhanced 2SFCA, passe 1/2 (demande)... lot 8/143
Enhanced 2SFCA, passe 1/2 (demande)... lot 9/143
Enhanced 2SFCA, passe 1/2 (demande)... lot 10/143
Enhanced 2SFCA, passe 1/2 (demande)... lot 11/143
Enhanced 2SFCA, passe 1/2 (demande)... lot 12/143
Enhanced 2SFCA, passe 1/2 (demande)... lot 13/143
Enhanced 2SFCA, passe 1/2 (demande)... lot 14/143
Enhanced 2SFCA, passe 1/2 (demande)... lot 15/143
Enhanced 2SFCA, passe 1/2 (demande)... lot 16/143
Enhanced 2SFCA, passe 1/2 (demande)... lot 17/143
Enhanced 2SFCA, passe 1/2 (demande)... lot 18/143
Enhanced 2SFCA, passe 1/2 (demande)... lot 19/143
Enhanced 2SFCA, passe 1/2 (demande)... lot 20/143
Enhanced 

,id,equipements
0,0,0.0
1,1,0.0
2,2,0.0
3,3,0.0
4,4,0.0


## 3.2.5 Comparaison des seuils <= vs < (`cutoff`)

Même illustration que le notebook source (`cumulative_cutoff()` compte les trajets `<= cutoff`, pas `< cutoff` — `cutoff=29` équivaut donc à un seuil "moins de 30 minutes"), appliquée à `opportunity="equipements"` directement sur `land_use_data` (pas de domaine BPE à choisir, cf. 3.2.2 ci-dessus).

In [14]:
cum_cutoff_30 = cumulative_cutoff(
    ttm,
    land_use_data=land_use_data,
    opportunity="equipements",
    travel_cost="travel_time",
    cutoff=30,
)

cum_cutoff_29 = cumulative_cutoff(
    ttm,
    land_use_data=land_use_data,
    opportunity="equipements",
    travel_cost="travel_time",
    cutoff=29,
)

cutoff_comparison = cum_cutoff_30.merge(
    cum_cutoff_29, on="id", suffixes=("_cutoff_30_inclus", "_cutoff_29_soit_moins_de_30min")
)

cutoff_comparison.head()

,id,equipements_cutoff_30_inclus,equipements_cutoff_29_soit_moins_de_30min
0,0,0,0
1,1,0,0
2,2,0,0
3,3,0,0
4,4,0,0


## Carte de la grille de population

In [15]:
os.makedirs(os.path.join(output_path, nom_reseau_str), exist_ok=True)
output_html_worldpop = os.path.join(output_path, nom_reseau_str, f"{NOM_ZONE_GTFS}_worldpop_gtfs_{MARGE_KM}km.html")

carte_pop = carte_population_worldpop(population_grid_agglo, NOM_ZONE_GTFS, ANNEE_GTFS, tiles=FONDS_CARTE[FOND_CARTE])
carte_pop.save(output_html_worldpop)
print(f"✓ Carte enregistrée : {output_html_worldpop}")

carte_pop

✓ Carte enregistrée : /Users/antoinechevre/Documents/_0_Programme/6_GTFS_universal/gtfs_analysis_app/output/AMUGA - Gbaka 10 à 14 places - Gbaka 18 à 22 places - Gbaka 26 à 32 places - Gba…/Abidjan_AMUGA_GTFS_2025_mapping_v2_worldpop_gtfs_5km.html


## Analyse accessibilité / pop par domaine + inégalités par niveau de vie (à faire plus tard)

Correspond à la section "analyse accessibilite / pop" du notebook source (9.1 temps d'accès au pôle le plus proche par domaine, 9.2 pôles accessibles à 30/45 min, 9.3 inégalités par décile de niveau de vie). Doublement bloqué pour la Côte d'Ivoire : boucle sur les domaines BPE (cf. ci-dessus) ET décile de niveau de vie calculé sur `ind_snv` (Filosofi, INSEE — France uniquement, pas de colonne équivalente dans la grille WorldPop).

## Tableaux récapitulatifs pôles d'équipements (à faire plus tard)

Correspond aux tableaux `pct_poles_atteignables_par_carreau`/`moyenne_ponderee_pct_poles` du notebook source (% de pôles d'équipements majeurs atteignables par domaine et par durée) — BPE requise.

## Sauvegarde index benchmark (à faire plus tard)

Correspond à `calculer_index_benchmark` du notebook source (indicateurs de synthèse par domaine BPE et par décile de niveau de vie, agrégés dans le CSV de benchmark inter-réseaux) — BPE et décile de niveau de vie requis, cf. sections précédentes.